# OpenTelemetry setup

In [2]:
from opentelemetry import trace
from opentelemetry.sdk.trace import TracerProvider
from opentelemetry.sdk.trace.export import ConsoleSpanExporter, SimpleSpanProcessor

provider = TracerProvider()
provider.add_span_processor(
    SimpleSpanProcessor(ConsoleSpanExporter())
)
trace.set_tracer_provider(provider)

tracer = trace.get_tracer("llm-zoomcamp")

In [5]:
from starter import index, client
from rag_helper import RAGBase

In [6]:
class RAGTraced(RAGBase):
    def rag(self, query):
        with tracer.start_as_current_span("rag"):
            return super().rag(query)

    def search(self, query, num_results=5):
        with tracer.start_as_current_span("search"):
            return super().search(query, num_results)

    def llm(self, prompt):
        with tracer.start_as_current_span("llm"):
            return super().llm(prompt)

# First trace

In [7]:
rag_traced = RAGTraced(index=index, llm_client=client)
answer = rag_traced.rag("How does the agentic loop keep calling the model until it stops?")
print(answer)

{
    "name": "search",
    "context": {
        "trace_id": "0x423b695f4726ea745c6803376f5680cd",
        "span_id": "0x29a4141e028e884e",
        "trace_state": "[]"
    },
    "kind": "SpanKind.INTERNAL",
    "parent_id": "0x1fa5fac5cd05b9a0",
    "start_time": "2026-07-20T17:06:22.777447Z",
    "end_time": "2026-07-20T17:06:22.779505Z",
    "status": {
        "status_code": "UNSET"
    },
    "attributes": {},
    "events": [],
    "links": [],
    "resource": {
        "attributes": {
            "telemetry.sdk.language": "python",
            "telemetry.sdk.name": "opentelemetry",
            "telemetry.sdk.version": "1.44.0",
            "service.instance.id": "89737347-b890-4f2b-ace9-bdc2c864fa12",
            "service.name": "unknown_service"
        },
        "schema_url": ""
    }
}
{
    "name": "llm",
    "context": {
        "trace_id": "0x423b695f4726ea745c6803376f5680cd",
        "span_id": "0x87c4926515e48c12",
        "trace_state": "[]"
    },
    "kind": "SpanKind

# Capturing metrics as span attributes

Modify `llm()` to capture usage.

In [10]:
class RAGTraced(RAGBase):
    def rag(self, query):
        with tracer.start_as_current_span("rag"):
            return super().rag(query)

    def search(self, query, num_results=5):
        with tracer.start_as_current_span("search"):
            return super().search(query, num_results)

    def llm(self, prompt):
        with tracer.start_as_current_span("llm") as span:
            response = super().llm(prompt)

            usage = response.usage
            span.set_attribute("input_tokens", usage.input_tokens)
            span.set_attribute("output_tokens", usage.output_tokens)

            cost = (
                usage.input_tokens * 0.25 / 1_000_000
                + usage.output_tokens * 2.00 / 1_000_000
            )
            span.set_attribute("cost", cost)

            return response

In [11]:
rag_traced = RAGTraced(index=index, llm_client=client)
answer = rag_traced.rag("How does the agentic loop keep calling the model until it stops?")
print(answer)

{
    "name": "search",
    "context": {
        "trace_id": "0xb7bcf6633b9f413038c59da39abfb0bb",
        "span_id": "0x0edee4384bd22a68",
        "trace_state": "[]"
    },
    "kind": "SpanKind.INTERNAL",
    "parent_id": "0x81a2036c43e0d914",
    "start_time": "2026-07-20T17:13:01.650795Z",
    "end_time": "2026-07-20T17:13:01.653018Z",
    "status": {
        "status_code": "UNSET"
    },
    "attributes": {},
    "events": [],
    "links": [],
    "resource": {
        "attributes": {
            "telemetry.sdk.language": "python",
            "telemetry.sdk.name": "opentelemetry",
            "telemetry.sdk.version": "1.44.0",
            "service.instance.id": "89737347-b890-4f2b-ace9-bdc2c864fa12",
            "service.name": "unknown_service"
        },
        "schema_url": ""
    }
}
{
    "name": "llm",
    "context": {
        "trace_id": "0xb7bcf6633b9f413038c59da39abfb0bb",
        "span_id": "0x7788eb9645bcbcd3",
        "trace_state": "[]"
    },
    "kind": "SpanKind

# Span timing

Times for first query.

In [12]:
from datetime import datetime

def duration_ms(start, end):
    fmt = "%Y-%m-%dT%H:%M:%S.%fZ"
    return (datetime.strptime(end, fmt) - datetime.strptime(start, fmt)).total_seconds() * 1000

spans = {
    "search": ("2026-07-20T17:06:22.777447Z", "2026-07-20T17:06:22.779505Z"),
    "llm":    ("2026-07-20T17:06:22.780312Z", "2026-07-20T17:06:24.531066Z"),
    "rag":    ("2026-07-20T17:06:22.777322Z", "2026-07-20T17:06:24.531489Z"),
}

for name, (start, end) in spans.items():
    print(f"{name:8} {duration_ms(start, end):9.2f} ms")

search        2.06 ms
llm        1750.75 ms
rag        1754.17 ms


# Saving traces to SQLite

In [1]:
import sqlite3
from opentelemetry import trace
from opentelemetry.sdk.trace import TracerProvider
from opentelemetry.sdk.trace.export import (
    SimpleSpanProcessor, SpanExporter, SpanExportResult
)


class SQLiteSpanExporter(SpanExporter):
    def __init__(self, db_path="traces.db"):
        self.conn = sqlite3.connect(db_path)
        self.conn.execute("""
            CREATE TABLE IF NOT EXISTS spans (
                name TEXT,
                start_time INTEGER,
                end_time INTEGER,
                input_tokens INTEGER,
                output_tokens INTEGER,
                cost REAL
            )
        """)
        self.conn.commit()

    def export(self, spans):
        for span in spans:
            attrs = dict(span.attributes or {})
            self.conn.execute(
                "INSERT INTO spans VALUES (?, ?, ?, ?, ?, ?)",
                (
                    span.name,
                    span.start_time,
                    span.end_time,
                    attrs.get("input_tokens"),
                    attrs.get("output_tokens"),
                    attrs.get("cost"),
                ),
            )
        self.conn.commit()
        return SpanExportResult.SUCCESS

    def shutdown(self):
        self.conn.close()

    def force_flush(self, timeout_millis=30000):
        return True


provider = TracerProvider()
provider.add_span_processor(SimpleSpanProcessor(SQLiteSpanExporter("traces.db")))
trace.set_tracer_provider(provider)
tracer = trace.get_tracer("llm-zoomcamp")

In [2]:
from starter import index, client
from rag_helper import RAGBase


class RAGTraced(RAGBase):
    def rag(self, query):
        with tracer.start_as_current_span("rag"):
            return super().rag(query)

    def search(self, query, num_results=5):
        with tracer.start_as_current_span("search"):
            return super().search(query, num_results)

    def llm(self, prompt):
        with tracer.start_as_current_span("llm") as span:
            response = super().llm(prompt)
            usage = response.usage
            span.set_attribute("input_tokens", usage.input_tokens)
            span.set_attribute("output_tokens", usage.output_tokens)
            cost = (
                usage.input_tokens * 0.25 / 1_000_000
                + usage.output_tokens * 2.00 / 1_000_000
            )
            span.set_attribute("cost", cost)
            return response

In [3]:
rag_traced = RAGTraced(index=index, llm_client=client)
answer = rag_traced.rag("How does the agentic loop keep calling the model until it stops?")
print(answer)

The loop keeps calling the model with a `while True` loop, and after each response it checks whether the model returned any `function_call` items.

- If there are function calls, the code runs the tool, appends the tool output to `messages`, and loops again.
- If there are no function calls, it breaks out of the loop and stops.

So the stop condition is:

```python
if has_function_calls == False:
    break
```

In short: it keeps going until the model returns a final message with no more tool calls.


In [4]:
import sqlite3
conn = sqlite3.connect("traces.db")
for row in conn.execute("SELECT name, input_tokens, output_tokens, cost FROM spans"):
    print(row)
conn.close()

('search', None, None, None)
('llm', 7111, 118, 0.00201375)
('rag', None, None, None)


# Querying trace data

In [5]:
rag_traced = RAGTraced(index=index, llm_client=client)
answer = rag_traced.rag("How does the agentic loop keep calling the model until it stops?")
print(answer)

The loop keeps calling the model with a `while True` loop. After each model response, it checks whether there were any `function_call` items. If there were, it runs the tools, appends the tool results to `messages`, and calls the model again.

It stops when the model returns a response with no function calls:

```python
if has_function_calls == False:
    break
```

So the model decides when it’s done, and the code keeps looping until there are no more tool calls.


In [6]:
import sqlite3
conn = sqlite3.connect("traces.db")
rows = conn.execute("""
    SELECT name,
           COUNT(*) AS calls,
           SUM(end_time - start_time) / 1e6 AS total_ms,
           AVG(end_time - start_time) / 1e6 AS avg_ms
    FROM spans
    WHERE name != 'rag'
    GROUP BY name
    ORDER BY total_ms DESC
""").fetchall()
for r in rows:
    print(f"{r[0]:8} calls={r[1]:3}  total={r[2]:9.2f} ms  avg={r[3]:8.2f} ms")
conn.close()

llm      calls=  2  total=  4814.66 ms  avg= 2407.33 ms
search   calls=  2  total=     4.10 ms  avg=    2.05 ms


# Token stability across runs

In [7]:
query = "How does the agentic loop keep calling the model until it stops?"
for _ in range(3):
    rag_traced.rag(query)

In [8]:
import sqlite3
import pandas as pd

conn = sqlite3.connect("traces.db")
df = pd.read_sql("SELECT * FROM spans WHERE name = 'llm'", conn)
conn.close()

print(df[["input_tokens", "output_tokens", "cost"]])
print()
print("min:", df.input_tokens.min())
print("max:", df.input_tokens.max())
print("spread:", (df.input_tokens.max() - df.input_tokens.min()) / df.input_tokens.min() * 100, "%")

   input_tokens  output_tokens      cost
0          7111            118  0.002014
1          7111            109  0.001996
2          7111             92  0.001962
3          7111            119  0.002016
4          7111             88  0.001954

min: 7111
max: 7111
spread: 0.0 %
